# 시드 고정

In [2]:
import os
import random
import numpy as np
import torch


def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(1)

# 데이터 처리

### 전역 설정
- 잠자는 시간, 깨어있는 시간, 일 하는 시간, 자유시간
- 공휴일

In [37]:
SLEEP_HOURS = tuple(range(0, 7))
ACTIVE_HOURS = tuple(range(7, 24))
WORK_HOURS = tuple(range(7, 19))
FREE_HOURS = tuple(range(19, 24))

HOLIDAY_DATES = [
    '2024-08-15',
    '2024-09-16',
    '2024-09-17',
    '2024-09-18',
    '2024-10-03',
    '2024-10-09',
]

### 유틸 함수

In [38]:
from pathlib import Path


DATA_DIR = Path("./data")

In [39]:
from enum import Enum


class DataType(Enum):
    mACStatus = "mACStatus"
    mActivity = "mActivity"
    mAmbience = "mAmbience"
    mBle = "mBle"
    mGps = "mGps"
    mLight = "mLight"
    mScreenStatus = "mScreenStatus"
    mUsageStats = "mUsageStats"
    mWifi = "mWifi"
    wHr = "wHr"
    wLight = "wLight"
    wPedo = "wPedo"

In [40]:
import pandas as pd


def load_data(data_type: DataType):
    file_path = DATA_DIR / f"ch2025_data_items/ch2025_{data_type.value}.parquet"
    df = pd.read_parquet(file_path)
    df["subject_id"] = df["subject_id"].astype("category")
    df["lifelog_date"] = df["timestamp"].dt.normalize()
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["hour"] = df["timestamp"].dt.hour
    df["minute"] = df["timestamp"].dt.minute
    df["weekday"] = df["timestamp"].dt.weekday

    fixed_columns = ["subject_id", "timestamp", "lifelog_date", "month", "day", "hour", "minute", "weekday"]
    columns = df.columns.tolist()
    columns = fixed_columns + [col for col in columns if col not in fixed_columns]
    df = df[columns]
    df = df.sort_values(by=["subject_id", "timestamp"])

    return df


def load_train():
    df = pd.read_csv(DATA_DIR / "ch2025_metrics_train.csv")
    df["subject_id"] = df["subject_id"].astype("category")
    df["sleep_date"] = pd.to_datetime(df["sleep_date"]).dt.normalize()
    df["lifelog_date"] = pd.to_datetime(df["lifelog_date"]).dt.normalize()
    return df


def load_val():
    from io import StringIO
    train_df = load_train()
    val_ids = "subject_id,sleep_date\nid01,2024-07-24\nid01,2024-07-27\nid01,2024-08-18\nid01,2024-08-19\nid01,2024-08-20\nid01,2024-08-21\nid01,2024-08-22\nid01,2024-08-24\nid01,2024-08-25\nid01,2024-08-26\nid01,2024-08-27\nid01,2024-08-28\nid01,2024-08-29\nid01,2024-08-30\nid02,2024-08-23\nid02,2024-08-24\nid02,2024-09-16\nid02,2024-09-17\nid02,2024-09-19\nid02,2024-09-20\nid02,2024-09-21\nid02,2024-09-22\nid02,2024-09-23\nid02,2024-09-24\nid02,2024-09-25\nid02,2024-09-26\nid02,2024-09-27\nid02,2024-09-28\nid03,2024-08-30\nid03,2024-09-01\nid03,2024-09-02\nid03,2024-09-03\nid03,2024-09-05\nid03,2024-09-06\nid03,2024-09-07\nid04,2024-09-03\nid04,2024-09-04\nid04,2024-09-05\nid04,2024-09-06\nid04,2024-09-07\nid04,2024-09-08\nid04,2024-09-09\nid04,2024-10-08\nid04,2024-10-09\nid04,2024-10-10\nid04,2024-10-11\nid04,2024-10-12\nid04,2024-10-13\nid04,2024-10-14\nid05,2024-10-19\nid05,2024-10-23\nid05,2024-10-24\nid05,2024-10-25\nid05,2024-10-26\nid05,2024-10-27\nid05,2024-10-28\nid06,2024-07-25\nid06,2024-07-26\nid06,2024-07-27\nid06,2024-07-28\nid06,2024-07-29\nid06,2024-07-30\nid06,2024-07-31\nid07,2024-07-07\nid07,2024-07-08\nid07,2024-07-09\nid07,2024-07-10\nid07,2024-07-11\nid07,2024-07-12\nid07,2024-07-13\nid07,2024-07-30\nid07,2024-08-01\nid07,2024-08-02\nid07,2024-08-03\nid07,2024-08-04\nid07,2024-08-05\nid07,2024-08-06\nid08,2024-08-28\nid08,2024-08-29\nid08,2024-08-30\nid08,2024-08-31\nid08,2024-09-01\nid08,2024-09-02\nid08,2024-09-04\nid09,2024-08-02\nid09,2024-08-22\nid09,2024-08-23\nid09,2024-08-24\nid09,2024-08-25\nid09,2024-08-27\nid09,2024-08-28\nid09,2024-08-29\nid09,2024-08-30\nid09,2024-08-31\nid09,2024-09-01\nid09,2024-09-02\nid09,2024-09-03\nid09,2024-09-04\nid10,2024-08-28\nid10,2024-08-30\nid10,2024-08-31\nid10,2024-09-01\nid10,2024-09-02\nid10,2024-09-03\nid10,2024-09-06\n"
    val_df = pd.read_csv(StringIO(val_ids))
    val_df = val_df.astype({"subject_id": "category", "sleep_date": "datetime64[ns]"})
    val_df = train_df.merge(val_df, on=["subject_id", "sleep_date"], how="inner")
    return val_df


def load_test():
    return pd.read_csv(DATA_DIR / "ch2025_metrics_test.csv")

In [57]:
def shift_lifelog_date(df, target_hours=SLEEP_HOURS):
    df = df.copy()
    mask = df["hour"].isin(target_hours) & df["hour"].lt(12)
    df.loc[mask, "lifelog_date"] = df.loc[mask, "lifelog_date"] - pd.Timedelta(days=1)
    df.loc[mask, "day"] = df.loc[mask, "day"] - 1
    df = df.sort_values(by=["subject_id", "lifelog_date", "timestamp"])
    return df

In [58]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.float_format', lambda x: '%0.4f' % x)

def describe_df(df):
    print(f"# shape:\n{df.shape}\n")
    print(f"# dtypes:\n{df.dtypes}\n")
    print(f"# head:\n{df.head(3)}\n")
    nan_stats = df.isna().sum().to_frame(name='missing_count')
    nan_stats['missing_ratio(%)'] = (df.isna().mean() * 100).round(2)
    print(f"# nan_stats:\n" + nan_stats.to_string() + "\n")

### ✔️ mACStatus 핸드폰 충전상태
- Indicates whether the smartphone is currently being charged.
- m_charging : 0/1 상태
- 핸드폰이 오랫 동안 충전했다는 의미?
 - 한 자리에 장시간 머물러 있었다.
 - 핸드폰을 장시간 사용하지 않았다.  

In [66]:
mACStatus_ori = load_data(DataType.mACStatus)
mACStatus_ori = shift_lifelog_date(mACStatus_ori, target_hours=SLEEP_HOURS)

len(mACStatus_ori), mACStatus_ori.head(1)

(939896,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday  m_charging
 0       id01 2024-06-26 12:03:00   2024-06-26      6   26    12       3        2           0)

In [67]:
def run_length_encoding(arr):
    """Run-Length Encoding"""
    if len(arr) == 0:
        return []

    diffs = np.diff(np.concatenate(([0], arr, [0])))
    run_starts = np.where(diffs == 1)[0]
    run_ends = np.where(diffs == -1)[0]
    return run_ends - run_starts

def process_mACStatus(df):
    status = df["m_charging"].values

    def _process_feature(status):
        if len(status) == 0:
            return 0., 0., 0., 0., 0.

        # charging 상태 비율, 합
        ratio_charging = status.mean()
        sum_charging = status.sum()

        # 상태전이 횟수
        transitions = (status[1:] != status[:-1]).sum()

        lengths = run_length_encoding(status)
        avg_charging_duration = np.mean(lengths) if len(lengths) > 0 else 0
        max_charging_duration = np.max(lengths) if len(lengths) > 0 else 0

        return ratio_charging, sum_charging, transitions, avg_charging_duration, max_charging_duration

    # 하루
    charging_ratio, charging_sum, chargning_transitions, avg_charging_duration, max_charging_duration = _process_feature(status)

    # 잠자는 시간대
    sleep_status = status[df["hour"].isin(SLEEP_HOURS)]
    sleep_charging_ratio, sleep_charging_sum, sleep_charging_transitions, sleep_avg_charging_duration, sleep_max_charging_duration = _process_feature(sleep_status)

    return pd.Series({
        'charging_ratio': charging_ratio,
        'charging_sum': charging_sum,
        'charging_transitions': chargning_transitions,
        'avg_charging_duration': avg_charging_duration,
        'max_charging_duration': max_charging_duration,
        'sleep_charging_ratio': sleep_charging_ratio,
        'sleep_charging_sum': sleep_charging_sum,
        'sleep_charging_transitions': sleep_charging_transitions,
        'sleep_avg_charging_duration': sleep_avg_charging_duration,
        'sleep_max_charging_duration': sleep_max_charging_duration,
    })

mACStatus = (
    mACStatus_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mACStatus)
    .reset_index(drop=True)
)
describe_df(mACStatus)

# shape:
(803, 12)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
charging_ratio                        float64
charging_sum                          float64
charging_transitions                  float64
avg_charging_duration                 float64
max_charging_duration                 float64
sleep_charging_ratio                  float64
sleep_charging_sum                    float64
sleep_charging_transitions            float64
sleep_avg_charging_duration           float64
sleep_max_charging_duration           float64
dtype: object

# head:
  subject_id lifelog_date  charging_ratio  charging_sum  charging_transitions  avg_charging_duration  max_charging_duration  sleep_charging_ratio  sleep_charging_sum  sleep_charging_transitions  sleep_avg_charging_duration  sleep_max_charging_duration
0       id01   2024-06-26          0.1626      179.0000               31.0000                11.1875                41.0000                0.07

C:\Users\lhj56\AppData\Local\Temp\ipykernel_3832\216347306.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mACStatus)


### ✔️ mActivity 추정행동
- Value calculated by the Google Activity Recognition API.
 - 0 : IN_VEHICLE
 - 1 : ON_BICYCLE
 - 2 : ON_FOOT
 - 3 : STILL (not moving)
 - 4 : UNKNOWN
 - 5 : TILTING (This often occurs when a device is picked up from a desk or a user who is sitting stands up.)
 - 7 : WALKING
 - 8 : RUNNING
- 근무시간   : 오전 7시부터 오후 6시까지
- 근무외시간 : 오후6시부터 12시까지

In [68]:
mActivity_ori = load_data(DataType.mActivity)
mActivity_ori = shift_lifelog_date(mActivity_ori, target_hours=SLEEP_HOURS)

len(mActivity_ori),mActivity_ori.head(1)

(961062,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday  m_activity
 0       id01 2024-06-26 12:03:00   2024-06-26      6   26    12       3        2           4)

In [69]:
def process_mActivity(df):
    activity = df["m_activity"].values.astype("int8")

    EXCLUDE_ACTIVITY = [3, 4]
    WALKING_ACTIVITY = [1, 2, 7, 8]
    VEHICLE_ACTIVITY = [0]

    def _process_feature(activity):
        if len(activity) == 0:
            return 0., 0., 0.
        
        # Walking minutes
        walking_minutes = np.isin(activity, WALKING_ACTIVITY).sum()

        # Vehicle minutes
        vehicle_minutes = np.isin(activity, VEHICLE_ACTIVITY).sum()

        # Activity minutes
        activity_minutes = (1 - np.isin(activity, EXCLUDE_ACTIVITY)).sum()

        return walking_minutes, vehicle_minutes, activity_minutes
        
    # 하루
    walking_minutes, vehicle_minutes, activity_minutes = _process_feature(activity)

    # 잠자는 시간대
    sleep_walking_minutes, sleep_vehicle_minutes, sleep_activity_minutes = _process_feature(activity[df["hour"].isin(SLEEP_HOURS)])

    
    return pd.Series({
        'walking_minutes': walking_minutes,
        'vehicle_minutes': vehicle_minutes,
        'activity_minutes': activity_minutes,
        'sleep_walking_minutes': sleep_walking_minutes,
        'sleep_vehicle_minutes': sleep_vehicle_minutes,
        'sleep_activity_minutes': sleep_activity_minutes,
    })

mActivity = (
    mActivity_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mActivity)
    .reset_index(drop=True)
)
describe_df(mActivity)

# shape:
(803, 8)

# dtypes:
subject_id                      category
lifelog_date              datetime64[ns]
walking_minutes                  float64
vehicle_minutes                  float64
activity_minutes                 float64
sleep_walking_minutes            float64
sleep_vehicle_minutes            float64
sleep_activity_minutes           float64
dtype: object

# head:
  subject_id lifelog_date  walking_minutes  vehicle_minutes  activity_minutes  sleep_walking_minutes  sleep_vehicle_minutes  sleep_activity_minutes
0       id01   2024-06-26          37.0000         116.0000          153.0000                 5.0000                27.0000                 32.0000
1       id01   2024-06-27          30.0000         213.0000          243.0000                 4.0000                29.0000                 33.0000
2       id01   2024-06-28          36.0000         145.0000          181.0000                 3.0000                13.0000                 16.0000

# nan_stats:
              

C:\Users\lhj56\AppData\Local\Temp\ipykernel_3832\2047524889.py:42: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mActivity)


### ✔️ mAmbience 추정주변소리
- Ambient sound identification labels and their respective probabilities.
- 무슨 소리가 난게 중요할까?
- 새벽에 무슨 소리던지 소리가 난게 중요한 걸까?
- 여러 가지 소리 중에 노이즈도 포함되어 있을까?

In [70]:
mAmbience_ori = load_data(DataType.mAmbience)
mAmbience_ori = shift_lifelog_date(mAmbience_ori, target_hours=SLEEP_HOURS)

len(mAmbience_ori), mAmbience_ori.head(1)

(476577,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                                                                                                                                                                                                   m_ambience
 0       id01 2024-06-26 13:00:10   2024-06-26      6   26    13       0        2  [[Music, 0.30902618], [Vehicle, 0.081680894], [Motor vehicle (road), 0.04035286], [Outside, urban or manmade, 0.037144363], [Outside, rural or natural, 0.032663062], [Car, 0.03199804], [Speech, 0.029806137], [Inside, large room or hall, 0.01684492], [Truck, 0.016206821], [Sound effect, 0.01591479]])

In [71]:
def process_mAmbience(df):
    ambience = df["m_ambience"].values  # [[label, prob], ...], [[label, prob], ...]

    def _process_feature(ambience):
        labels = []

        for amb in ambience:
            labels_, _ = zip(*amb)
            labels.extend(labels_)
        
        unique_label_count = len(labels)
        snor_count = len(list(filter(lambda x: "snor" in x.lower(), labels)))

        return unique_label_count, snor_count
    
    # 활동시간
    active_hour_unique_label_count, active_hour_snor_count = _process_feature(ambience[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는시간
    sleep_hour_unique_label_count, sleep_hour_snor_count = _process_feature(ambience[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_unique_label_count': active_hour_unique_label_count,
        'active_hour_snor_count': active_hour_snor_count,
        'sleep_hour_unique_label_count': sleep_hour_unique_label_count,
        'sleep_hour_snor_count': sleep_hour_snor_count,
    })


mAmbience = (
    mAmbience_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mAmbience)
    .reset_index(drop=True)
)
describe_df(mAmbience)

# shape:
(803, 6)

# dtypes:
subject_id                              category
lifelog_date                      datetime64[ns]
active_hour_unique_label_count             int64
active_hour_snor_count                     int64
sleep_hour_unique_label_count              int64
sleep_hour_snor_count                      int64
dtype: object

# head:
  subject_id lifelog_date  active_hour_unique_label_count  active_hour_snor_count  sleep_hour_unique_label_count  sleep_hour_snor_count
0       id01   2024-06-26                            3040                       4                           2100                      0
1       id01   2024-06-27                            5100                       0                           2100                      0
2       id01   2024-06-28                            5000                       0                           2100                      0

# nan_stats:
                                missing_count  missing_ratio(%)
subject_id                      

C:\Users\lhj56\AppData\Local\Temp\ipykernel_3832\1264552170.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mAmbience)


### ✔️ mBle 블루투스
- Bluetooth devices around individual subject.
 - 7936 : Wearable, Headset, AV Device
 - 1796 : Peripheral (입력장치) 계열
 - 0 : 정보 없음 또는 알 수 없음(Unknown)
 - 1084 : Audio/Video (스피커, 헤드셋, 이어폰, TV 등)
 - 524 : Phone (휴대폰, 스마트폰)
 - 1060 : Headphones
 - 284 : commputer (PC, 노트북, PDA)

In [72]:
mBle_ori = load_data(DataType.mBle)
mBle_ori = shift_lifelog_date(mBle_ori, target_hours=SLEEP_HOURS)

len(mBle_ori), mBle_ori.head(1)

(21830,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [73]:
def process_mBle(df):
    ble = df["m_ble"].values  # [[{"address": "xx:xx:xx:xx:xx:xx", "device_class": "0", "rssi": -70}, ...], [...], ...]

    def _process_feature(ble):
        if len(ble) == 0:
            return 0., 0., 0., 0., 0.
        
        rssi = []
        devices = []
        for ble_data in ble:
            for device in ble_data:
                rssi.append(device["rssi"])
                devices.append(device["device_class"])

        rssi = np.array(rssi)
        rssi_mean = rssi.mean() if len(rssi) > 0 else 0
        rssi_min = rssi.min() if len(rssi) > 0 else 0
        rssi_max = rssi.max() if len(rssi) > 0 else 0

        unknown_count = devices.count("0")
        others_count = len(devices) - unknown_count
        others_ratio = others_count / len(devices) if len(devices) > 0 else 0
        unknown_ratio = unknown_count / len(devices) if len(devices) > 0 else 0

        return rssi_mean, rssi_min, rssi_max, others_ratio, unknown_ratio
    
    # 일할때
    work_hour_rssi_mean, work_hour_rssi_min, work_hour_rssi_max, work_hour_others_ratio, work_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(WORK_HOURS)])

    # 퇴근후
    free_hour_rssi_mean, free_hour_rssi_min, free_hour_rssi_max, free_hour_others_ratio, free_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(FREE_HOURS)])

    # 잠자는시간
    sleep_hour_rssi_mean, sleep_hour_rssi_min, sleep_hour_rssi_max, sleep_hour_others_ratio, sleep_hour_unknown_ratio = _process_feature(ble[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'work_hour_rssi_mean': work_hour_rssi_mean,
        'work_hour_rssi_min': work_hour_rssi_min,
        'work_hour_rssi_max': work_hour_rssi_max,
        'work_hour_others_ratio': work_hour_others_ratio,
        'work_hour_unknown_ratio': work_hour_unknown_ratio,
        'free_hour_rssi_mean': free_hour_rssi_mean,
        'free_hour_rssi_min': free_hour_rssi_min,
        'free_hour_rssi_max': free_hour_rssi_max,
        'free_hour_others_ratio': free_hour_others_ratio,
        'free_hour_unknown_ratio': free_hour_unknown_ratio,
        'sleep_hour_rssi_mean': sleep_hour_rssi_mean,
        'sleep_hour_rssi_min': sleep_hour_rssi_min,
        'sleep_hour_rssi_max': sleep_hour_rssi_max,
        'sleep_hour_others_ratio': sleep_hour_others_ratio,
        'sleep_hour_unknown_ratio': sleep_hour_unknown_ratio
    })
mBle = (
    mBle_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mBle)
    .reset_index(drop=True)
)
describe_df(mBle)


# shape:
(726, 17)

# dtypes:
subject_id                        category
lifelog_date                datetime64[ns]
work_hour_rssi_mean                float64
work_hour_rssi_min                 float64
work_hour_rssi_max                 float64
work_hour_others_ratio             float64
work_hour_unknown_ratio            float64
free_hour_rssi_mean                float64
free_hour_rssi_min                 float64
free_hour_rssi_max                 float64
free_hour_others_ratio             float64
free_hour_unknown_ratio            float64
sleep_hour_rssi_mean               float64
sleep_hour_rssi_min                float64
sleep_hour_rssi_max                float64
sleep_hour_others_ratio            float64
sleep_hour_unknown_ratio           float64
dtype: object

# head:
  subject_id lifelog_date  work_hour_rssi_mean  work_hour_rssi_min  work_hour_rssi_max  work_hour_others_ratio  work_hour_unknown_ratio  free_hour_rssi_mean  free_hour_rssi_min  free_hour_rssi_max  free_hour_others_r

C:\Users\lhj56\AppData\Local\Temp\ipykernel_3832\4233196545.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mBle)


### ✔️ mGps, GPS 기반 핸드폰 위치
- Multiple GPS coordinates measured within a single minute using the smartphone.
- speed가 1보다 큰경우 정지 상태가 아니고 움직이고 있다고 판단
 - 0.5-2 : 걸어서 이동하는 경우  
 - 2-5 : 조깅
 - 5 이상 : 차를 타고 이동하는 경우

- speed가 0.5-2사이를 하루에 몇분동안 지속했는지?
- speed가 2-5사이를 하루에 몇분동안 지속했는지? (유산소 운동 시간)
- speed가 5이상을 하루에 몇분동안 지속했는지?  

In [74]:
mGps_ori = load_data(DataType.mGps)
mGps_ori = shift_lifelog_date(mGps_ori, target_hours=SLEEP_HOURS)

len(mGps_ori), mGps_ori.head(1)

(800611,
   subject_id           timestamp lifelog_date  month  day  hour  minute  weekday                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              

In [95]:
def haversine_np(lon1, lat1, lon2, lat2, radius=6371):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    return radius * c

def process_mGps(df):
    gps = df["m_gps"].values  # [[{'altitude': 110.6, 'latitude': 0.2077385, 'longitude': 0.170027, 'speed': 0.0}, ...], ...]
    timestamps = df["timestamp"].values

    def _process_feature(gps, timestamps):
        if len(gps) == 0:
            return 0., 0., 0., 0., 0., 0., 0.

        # n-분 단위
        latitudes = []
        longitudes = []
        altitudes = []
        speeds = []
        minutes = []  # 누적 분

        for i, (gps_data, timestamp) in enumerate(zip(gps, timestamps)):
            _latitudes = []
            _longitudes = []
            _altitudes = []
            _speeds = []
            for data in gps_data:
                _latitudes.append(data["latitude"])
                _longitudes.append(data["longitude"])
                _altitudes.append(data["altitude"])
                _speeds.append(data["speed"])
            
            latitudes.append(np.mean(_latitudes))
            longitudes.append(np.mean(_longitudes))
            altitudes.append(np.mean(_altitudes))
            speeds.append(np.mean(_speeds))
            minutes.append(1 if i == 0 else pd.Timedelta(timestamps[i] - timestamps[i-1]).total_seconds() / 60)

        latitudes = np.array(latitudes)
        longitudes = np.array(longitudes)
        altitudes = np.array(altitudes)
        speeds = np.array(speeds)
        minutes = np.array(minutes)

        walk_minutes = minutes[(speeds >= 0.5) & (speeds < 2.0)].sum()
        jog_minutes = minutes[(2.0 <= speeds) & (speeds < 5.0)].sum()
        vehicle_minutes = minutes[(5.0 <= speeds)].sum()

        # 속도
        mean_speed = speeds.mean() if len(speeds) > 0 else 0
        max_speed = speeds.max() if len(speeds) > 0 else 0
        min_speed = speeds.min() if len(speeds) > 0 else 0

        # 이동거리
        distance = haversine_np(longitudes[:-1], latitudes[:-1], longitudes[1:], latitudes[1:]).sum()

        return walk_minutes, jog_minutes, vehicle_minutes, mean_speed, max_speed, min_speed, distance

    # 하루
    active_hour_walk_minutes, active_hour_jog_minutes, active_hour_vehicle_minutes, active_hour_mean_speed, active_hour_max_speed, active_hour_min_speed, active_hour_distance = _process_feature(gps[df["hour"].isin(ACTIVE_HOURS)], timestamps[df["hour"].isin(ACTIVE_HOURS)])

    # 잠자는 시간대
    sleep_hour_walk_minutes, sleep_hour_jog_minutes, sleep_hour_vehicle_minutes, sleep_hour_mean_speed, sleep_hour_max_speed, sleep_hour_min_speed, sleep_hour_distance = _process_feature(gps[df["hour"].isin(SLEEP_HOURS)], timestamps[df["hour"].isin(SLEEP_HOURS)])

    return pd.Series({
        'active_hour_walk_minutes': active_hour_walk_minutes,
        'active_hour_jog_minutes': active_hour_jog_minutes,
        'active_hour_vehicle_minutes': active_hour_vehicle_minutes,
        'active_hour_mean_speed': active_hour_mean_speed,
        'active_hour_max_speed': active_hour_max_speed,
        'active_hour_min_speed': active_hour_min_speed,
        'active_hour_distance': active_hour_distance,
        'exercise_flag': 1 if active_hour_jog_minutes > 10 else 0,  # n분 이상 조깅한 경우
        'sleep_hour_walk_minutes': sleep_hour_walk_minutes,
        'sleep_hour_jog_minutes': sleep_hour_jog_minutes,
        'sleep_hour_vehicle_minutes': sleep_hour_vehicle_minutes,
        'sleep_hour_mean_speed': sleep_hour_mean_speed,
        'sleep_hour_max_speed': sleep_hour_max_speed,
        'sleep_hour_min_speed': sleep_hour_min_speed,
        'sleep_hour_distance': sleep_hour_distance
    })

mGps = (
    mGps_ori
    .groupby(["subject_id", "lifelog_date"], group_keys=False, as_index=False, sort=False, observed=True)
    .apply(process_mGps)
    .reset_index(drop=True)
)
describe_df(mGps)


# shape:
(759, 17)

# dtypes:
subject_id                           category
lifelog_date                   datetime64[ns]
active_hour_walk_minutes              float64
active_hour_jog_minutes               float64
active_hour_vehicle_minutes           float64
active_hour_mean_speed                float64
active_hour_max_speed                 float64
active_hour_min_speed                 float64
active_hour_distance                  float64
exercise_flag                         float64
sleep_hour_walk_minutes               float64
sleep_hour_jog_minutes                float64
sleep_hour_vehicle_minutes            float64
sleep_hour_mean_speed                 float64
sleep_hour_max_speed                  float64
sleep_hour_min_speed                  float64
sleep_hour_distance                   float64
dtype: object

# head:
  subject_id lifelog_date  active_hour_walk_minutes  active_hour_jog_minutes  active_hour_vehicle_minutes  active_hour_mean_speed  active_hour_max_speed  active_hour

C:\Users\lhj56\AppData\Local\Temp\ipykernel_3832\3521667569.py:93: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(process_mGps)


In [96]:
mGps["exercise_flag"].sum() / len(mGps)  # 전체 중 운동한 날 비율

np.float64(0.6271409749670619)